### Purpose
The purpose of this notebook is to facilitation experimentation and verification before implementation.

<div style="height: 8px; background-color: blue; border-radius: 4px; margin: 20px 0;"></div>

#### Step 1 - Data Acquisition

In [ ]:
from pathlib import Path
import requests
from zipfile import ZipFile
import hashlib


PROJECT_ROOT = Path.cwd().parent
RAW_PATH = PROJECT_ROOT / 'raw'
ZIP_PATH = RAW_PATH / 'ml-100k.zip'

URL = 'https://files.grouplens.org/datasets/movielens/ml-100k.zip'

<b>Download ZIP, run an integrity check, unpack it, create a SHA256 hash.</b>

In [ ]:
RAW_PATH.mkdir(parents=True, exist_ok=True)

if not ZIP_PATH.exists():
    print(f'Downloading MoveLens 100k data to {ZIP_PATH}')
    response = requests.get(URL, timeout=120)
    response.raise_for_status()
    ZIP_PATH.write_bytes(response.content)
    print('Download complete.')
else:
    print(f'ZIP already exists at {ZIP_PATH}. Skipping download.')


with ZipFile(ZIP_PATH) as archive:
    corrupt_file = archive.testzip()

    if corrupt_file is not None:
        raise ValueError(f'ZIP integrity check failed: {corrupt_file}')
    print('ZIP integrity check succeeded.')

    archive.extractall(RAW_PATH)
    print(f'Dataset extracted to {RAW_PATH}')

sha256 = hashlib.sha256(ZIP_PATH.read_bytes()).hexdigest()
print(f'SHA256 hash is: {sha256}')

<div style="height: 8px; background-color: blue; border-radius: 4px; margin: 20px 0;"></div>

#### Step 0 - Describe Raw Data

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


PROJECT_ROOT = Path.cwd().parent
DATA_PATH = PROJECT_ROOT / 'raw' / 'ml-100k'

FIGURES_PATH = DATA_PATH.parent.parent / 'figures'
FIGURES_PATH.mkdir(parents=True, exist_ok=True)

<div style="height: 2px; background-color: blue; border-radius: 4px; margin: 20px 0;"></div>
<b>u.user</b>

For the u.user dataset, the shape is (943, 5) and each row represents a user and the data is tidy. 
The README leaves out the fact that the zip code should be loaded as a string to preserve leading zeros.
The README does not specify types.

In [ ]:
users = pd.read_csv(DATA_PATH / 'u.user', sep='|', header=None, names=['user_id', 'age', 'gender', 'occupation', 'zip_code'], 
                   dtype={'zip_code': 'string'})

In [ ]:
users.info()

<div style="height: 2px; background-color: blue; border-radius: 4px; margin: 20px 0;"></div>
<b>u.item</b>

For the u.item dataset, the shape is (1682, 24). Each row represents a movie and the data is not tidy because the 
values of what should be genre are represented as their own individual columns. The README incorrectly states that the data is tab delimited when, in fact, it is pipe delimited. The README also left out the fact that the encoding used is Latin-1. The README does not specify types. There are also missing values across release date, video release date and IMDb URL.

In [ ]:
item_base = ['movie id', 'movie title', 'release date', 'video release date', 'IMDb URL']

item_genres = ['unknown', 'Action', 'Adventure', 'Animation', "Children's", 'Comedy', 'Crime', 'Documentary', 'Drama', 'Fantasy',
               'Film-Noir', 'Horror', 'Musical', 'Mystery', 'Romance','Sci-Fi', 'Thriller', 'War', 'Western'
              ]

item_columns = item_base + item_genres

items = pd.read_csv(DATA_PATH / 'u.item', sep='|', header=None, names=item_columns, encoding='latin-1')

In [ ]:
items.info()

<div style="height: 2px; background-color: blue; border-radius: 4px; margin: 20px 0;"></div>
<b>u.data</b>

For the u.data dataset, the shape is (100000, 4). Each row represents a user rating for a movie. The data is tidy. The README does not specify types beyond the timestamp being epoch time.

In [ ]:
data = pd.read_csv(DATA_PATH / 'u.data', sep='\t', header=None, names=['user id', 'item id', 'rating', 'timestamp'])

In [ ]:
data.info()

<div style="height: 2px; background-color: blue; border-radius: 4px; margin: 20px 0;"></div>
<b>Chart 1 - Distribution of ratings </b>

In [ ]:
sns.countplot(data=data, x='rating')

plt.title('Distribution of Movie Ratings')
plt.xlabel('Rating')
plt.ylabel('Number of Ratings')
plt.savefig(FIGURES_PATH / 'rating_distribution.png')
plt.show()

This chart shows us the distribution of movie ratings.
This chart suggests that ratings are skewed positively. It may be worth investigating if any groups are driving this.

<div style="height: 2px; background-color: blue; border-radius: 4px; margin: 20px 0;"></div>
<b>Chart 2 - Number of Users by Occupation </b>

In [ ]:
plt.figure(figsize=(12, 6))

sns.countplot(data=users, y='occupation', order=users['occupation'].value_counts().index)

plt.title('Number of Users by Occupation')
plt.xlabel('Number of Users')
plt.ylabel('Occupation')
plt.savefig(FIGURES_PATH / 'users_by_occupation.png', dpi=150)
plt.show()

This chart shows the distribution of MovieLens users across occupations.
It can answer questions around if any particular group is overrepresented in the data. In this case, students are.